# 06 – Publish weights + deploy the browser app
1. Uploads the two LoRA adapters to the Hugging Face Hub (public download link for the report). Works on a CPU runtime.
2. Optional: runs the app on a GPU runtime with a temporary public link (fast; useful for recording the demo).

The app is normally run locally with `./run_app.sh` (Hugging Face no longer hosts Gradio Spaces for free).

Needs a Colab secret **HF_TOKEN**: huggingface.co → Settings → Access Tokens → *Write* token.

In [ ]:
# ==== EDIT THESE TWO LINES ====
GITHUB_USER = "FatimaRana50"
GIT_NAME    = "Fatima Rana"
# ==============================
REPO = "trialmatch"

!pip -q install peft datasets
!pip -q uninstall -y torchao       # Colab ships torchao 0.10; peft refuses to load with it (we do not use it)
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
token = userdata.get('GH_TOKEN')          # Colab sidebar -> key icon -> add secret GH_TOKEN
if not os.path.exists(f'/content/{REPO}'):
    !git clone -q https://{token}@github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!git pull -q
!git config user.name "{GIT_NAME}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
import sys; sys.path.insert(0, '/content/trialmatch/src')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
HF_USER = "f777777777"          # <- your Hugging Face username
TARGET_DIR = "/content/drive/MyDrive/trialmatch/lora_r16/final_adapter"
DRAFT_DIR  = "/content/drive/MyDrive/trialmatch/draft_0.5b/final_adapter"
TARGET_REPO = f"{HF_USER}/trialmatch-qwen2.5-1.5b-lora"
DRAFT_REPO  = f"{HF_USER}/trialmatch-qwen2.5-0.5b-draft-lora"
!pip -q install -U huggingface_hub
from huggingface_hub import HfApi
api = HfApi(token=userdata.get("HF_TOKEN"))
print("logged in as", api.whoami()["name"])

## 1. Upload adapters

In [ ]:
card = lambda base, what: f"""---
base_model: {base}
library_name: peft
license: apache-2.0
tags: [lora, clinical-trials, trialgpt]
---
# {what}
LoRA adapter for `{base}` trained on the training split (37 patients) of
`ncbi/TrialGPT-Criterion-Annotations` (patient-grouped split, seed 42).
Code, split manifest and evaluation: https://github.com/{GITHUB_USER}/trialmatch
**Educational screening prototype. Not for clinical use.**
"""
for d, repo, base, what in [(TARGET_DIR, TARGET_REPO, "Qwen/Qwen2.5-1.5B-Instruct", "TrialMatch criterion classifier (target)"),
                            (DRAFT_DIR, DRAFT_REPO, "Qwen/Qwen2.5-0.5B-Instruct", "TrialMatch draft model for speculative decoding")]:
    api.create_repo(repo, exist_ok=True)
    open(f"{d}/README.md", "w").write(card(base, what))
    api.upload_folder(folder_path=d, repo_id=repo, commit_message="Upload LoRA adapter")
    print("https://huggingface.co/" + repo)

## 2. (Optional, needs a GPU runtime) Run the app here with a temporary public link
Open the `https://....gradio.live` link printed below. Stop the cell to shut it down.

In [ ]:
!pip -q install gradio
%env TRIALMATCH_ADAPTER=/content/drive/MyDrive/trialmatch/lora_r16/final_adapter
%env TRIALMATCH_DRAFT_ADAPTER=/content/drive/MyDrive/trialmatch/draft_0.5b/final_adapter
%env TRIALMATCH_SHARE=1
!python app/app.py